# Embedding MRL — Colab runner

Chọn một method và model bên dưới rồi chạy **Runtime → Run all**. Notebook luôn đồng bộ branch mới nhất trước khi train, dùng `data/train/final_data.csv`, chỉ evaluate một lần trên test sau training, và ghi artifacts trực tiếp vào Google Drive.

> Với pair-classification, threshold được tune trực tiếp trên chính test set theo evaluator của repo. Đây là protocol được yêu cầu, nhưng cần mô tả rõ trong paper vì kết quả không phải held-out threshold selection.


## 1. Settings

In [2]:
#@title Experiment settings { display-mode: "form" }

METHOD = "gsr"  #@param ["mrl", "ese", "mipic", "gsr"]
MODEL = "bert"  #@param ["bert", "tinybert_6l", "bgem3", "qwen3_0.6b"]

EPOCHS = 5  #@param {type:"integer"}
BATCH_SIZE = 128  #@param {type:"integer"}
EVAL_BATCH_SIZE = 64  #@param {type:"integer"}
LEARNING_RATE = 2e-5  #@param {type:"number"}
MAX_LENGTH = 256  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
FP16 = False  #@param {type:"boolean"}
MAX_GRAD_NORM = 0.0  #@param {type:"number"}

# Chỉ dùng khi METHOD=gsr. Phải nhỏ hơn EPOCHS nếu GSR_WEIGHT > 0.
GSR_WEIGHT = 0.1  #@param {type:"number"}
GSR_WARMUP_EPOCHS = 1  #@param {type:"integer"}
GSR_TEACHER_BATCH_SIZE = 64  #@param {type:"integer"}

REPO_URL = "https://github.com/duncan-nguyen/embedding-mrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
RUN_NAME = ""  #@param {type:"string"}

# BGE-M3/Qwen3 thường cần batch 4–8 trên GPU 16 GB.
assert EPOCHS > 0 and BATCH_SIZE > 0 and EVAL_BATCH_SIZE > 0
if METHOD == "gsr" and GSR_WEIGHT > 0:
    assert 0 <= GSR_WARMUP_EPOCHS < EPOCHS, (
        "GSR_WARMUP_EPOCHS phải nằm trong [0, EPOCHS)."
    )
print(f"Selected: {METHOD.upper()} / {MODEL}")

Selected: GSR / bert


## 2. Mount Drive, sync code mới nhất và cài package

In [6]:
import os
import json
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request
import zipfile
from datetime import datetime
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/[Research Space]/[ICLR] Embedding MRL")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
default_name = f"{METHOD}_{MODEL}_{timestamp}"
base_name = re.sub(r"[^A-Za-z0-9._-]+", "_", RUN_NAME.strip() or default_name)
safe_name = base_name
suffix = 1
while (DRIVE_ROOT / safe_name).exists():
    safe_name = f"{base_name}_{suffix:02d}"
    suffix += 1
RUN_DIR = DRIVE_ROOT / safe_name
RUN_DIR.mkdir(parents=True, exist_ok=False)
CONSOLE_LOG = RUN_DIR / "console.log"
REPO_DIR = Path("/content/embedding-mrl-runtime")

def run(command, *, cwd=None, log_path=CONSOLE_LOG):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    with Path(log_path).open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(command) + "\n")
        process = subprocess.Popen(
            command, cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def download_public_snapshot(repo_url, branch, destination):
    match = re.fullmatch(
        r"https://github\.com/([^/]+)/([^/]+?)(?:\.git)?/?", repo_url.strip()
    )
    if not match:
        raise ValueError("REPO_URL phải có dạng https://github.com/owner/repo.git")
    owner, repo = match.groups()
    encoded_branch = urllib.parse.quote(branch, safe="")
    api_url = f"https://api.github.com/repos/{owner}/{repo}/commits/{encoded_branch}"
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "embedding-mrl-colab"}

    print(f"$ GET {api_url}")
    with urllib.request.urlopen(
        urllib.request.Request(api_url, headers=headers), timeout=60
    ) as response:
        commit = json.load(response)["sha"]

    archive_url = f"https://codeload.github.com/{owner}/{repo}/zip/{commit}"
    print(f"$ GET {archive_url}")
    with tempfile.TemporaryDirectory(prefix="embedding-mrl-", dir="/content") as temp_dir:
        temp_dir = Path(temp_dir)
        archive_path = temp_dir / "source.zip"
        with urllib.request.urlopen(
            urllib.request.Request(archive_url, headers=headers), timeout=180
        ) as response, archive_path.open("wb") as output:
            shutil.copyfileobj(response, output)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(temp_dir)
        extracted = [path for path in temp_dir.iterdir() if path.is_dir()]
        if len(extracted) != 1:
            raise RuntimeError(f"Archive GitHub không hợp lệ: {extracted}")
        if destination.exists():
            shutil.rmtree(destination)
        shutil.move(str(extracted[0]), str(destination))
    return commit

# Git transport trong một số Colab runtime có thể giữ credential/proxy lỗi.
# Tải snapshot public theo SHA để không cần git và vẫn tái lập được thí nghiệm.
commit = download_public_snapshot(REPO_URL, BRANCH, REPO_DIR)

run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR)
(RUN_DIR / "git_commit.txt").write_text(commit + "\n", encoding="utf-8")
print(f"\nCommit: {commit}")
print(f"Artifacts: {RUN_DIR}")
try:
    run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
except (FileNotFoundError, subprocess.CalledProcessError):
    print("WARNING: Không phát hiện GPU. Chọn Runtime → Change runtime type → GPU.")

Mounted at /content/drive
$ GET https://api.github.com/repos/duncan-nguyen/embedding-mrl/commits/main
$ GET https://codeload.github.com/duncan-nguyen/embedding-mrl/zip/b349fe5710a6be49d38b26370f272d3ed9c1385d
$ /usr/bin/python3 -m pip install -q -e .

Commit: b349fe5710a6be49d38b26370f272d3ed9c1385d
Artifacts: /content/drive/MyDrive/[Research Space]/[ICLR] Embedding MRL/gsr_bert_20260902_134835
$ nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
NVIDIA A100-SXM4-80GB, 81920 MiB


## 3. Resolve config

Các protocol quan trọng được khóa tại đây: full `final_data.csv`, test split, evaluation bật, và `eval.every_epoch=false`.

In [7]:
CONFIG_PATH = REPO_DIR / "configs" / METHOD / f"{MODEL}.yaml"
if not CONFIG_PATH.exists():
    available = sorted(
        str(path.relative_to(REPO_DIR))
        for path in (REPO_DIR / "configs").glob("*/*.yaml")
    )
    raise FileNotFoundError(f"Không có {CONFIG_PATH}. Available: {available}")

def yaml_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if value is None:
        return "null"
    if isinstance(value, float):
        text = repr(value)
        if "e" in text.lower():
            mantissa, exponent = text.lower().split("e")
            if "." not in mantissa:
                mantissa += ".0"
            if not exponent.startswith(("+", "-")):
                exponent = "+" + exponent
            text = f"{mantissa}e{exponent}"
        return text
    return str(value)

overrides = [
    f"name={safe_name}",
    f"train.output_dir={RUN_DIR}",
    f"train.epochs={EPOCHS}",
    f"train.batch_size={BATCH_SIZE}",
    f"train.lr={yaml_value(float(LEARNING_RATE))}",
    f"train.seed={SEED}",
    f"train.fp16={yaml_value(FP16)}",
    f"train.max_grad_norm={yaml_value(MAX_GRAD_NORM if MAX_GRAD_NORM > 0 else None)}",
    f"data.max_length={MAX_LENGTH}",
    "data.train_file=train/final_data.csv",
    "data.max_train_samples=null",
    "eval.enabled=true",
    "eval.split=test",
    "eval.every_epoch=false",
    f"eval.batch_size={EVAL_BATCH_SIZE}",
]
if METHOD == "gsr":
    overrides.extend([
        f"gsr.weight={yaml_value(float(GSR_WEIGHT))}",
        f"gsr.warmup_epochs={GSR_WARMUP_EPOCHS}",
        f"gsr.teacher_batch_size={GSR_TEACHER_BATCH_SIZE}",
    ])

TRAIN_ARGS = [item for override in overrides for item in ("--set", override)]
print("Locked protocol:")
print("  train data    = data/train/final_data.csv (full)")
print("  eval           = once after training, split=test")
print("  pair threshold = tuned on test by evaluator")
print("  output         =", RUN_DIR)
run(
    [sys.executable, "scripts/train.py", "--config", CONFIG_PATH,
     *TRAIN_ARGS, "--print-config"],
    cwd=REPO_DIR,
)

Locked protocol:
  train data    = data/train/final_data.csv (full)
  eval           = once after training, split=test
  pair threshold = tuned on test by evaluator
  output         = /content/drive/MyDrive/[Research Space]/[ICLR] Embedding MRL/gsr_bert_20260902_134835
$ /usr/bin/python3 scripts/train.py --config /content/embedding-mrl-runtime/configs/gsr/bert.yaml --set name=gsr_bert_20260902_134835 --set train.output_dir=/content/drive/MyDrive/[Research Space]/[ICLR] Embedding MRL/gsr_bert_20260902_134835 --set train.epochs=5 --set train.batch_size=128 --set train.lr=2.0e-05 --set train.seed=42 --set train.fp16=false --set train.max_grad_norm=null --set data.max_length=256 --set data.train_file=train/final_data.csv --set data.max_train_samples=null --set eval.enabled=true --set eval.split=test --set eval.every_epoch=false --set eval.batch_size=64 --set gsr.weight=0.1 --set gsr.warmup_epochs=1 --set gsr.teacher_batch_size=64 --print-config
name: gsr_bert_20260902_134835
method: gsr
mo

## 4. Train và evaluate một lần

In [8]:
import time

started = time.time()
run(
    [sys.executable, "scripts/train.py", "--config", CONFIG_PATH, *TRAIN_ARGS],
    cwd=REPO_DIR,
)
print(f"\nHoàn tất sau {(time.time() - started) / 60:.1f} phút")
print("Console log:", CONSOLE_LOG)
print("Trainer log:", RUN_DIR / "train.log")

$ /usr/bin/python3 scripts/train.py --config /content/embedding-mrl-runtime/configs/gsr/bert.yaml --set name=gsr_bert_20260902_134835 --set train.output_dir=/content/drive/MyDrive/[Research Space]/[ICLR] Embedding MRL/gsr_bert_20260902_134835 --set train.epochs=5 --set train.batch_size=128 --set train.lr=2.0e-05 --set train.seed=42 --set train.fp16=false --set train.max_grad_norm=null --set data.max_length=256 --set data.train_file=train/final_data.csv --set data.max_train_samples=null --set eval.enabled=true --set eval.split=test --set eval.every_epoch=false --set eval.batch_size=64 --set gsr.weight=0.1 --set gsr.warmup_epochs=1 --set gsr.teacher_batch_size=64
13:51:26 | INFO    | Experiment gsr_bert_20260902_134835 (method=gsr)
13:51:41 | INFO    | NumExpr defaulting to 12 threads.
13:51:44 | INFO    | Loading google-bert/bert-base-uncased
13:51:44 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faste

## 5. Kết quả

In [9]:
import json
import pandas as pd
from IPython.display import display

report = json.loads((RUN_DIR / "results.json").read_text(encoding="utf-8"))
scores = pd.read_csv(RUN_DIR / "results.csv").set_index("dim")
print(f"{report['experiment']['method'].upper()} / {report['experiment']['model']}")
print("Commit:", (RUN_DIR / "git_commit.txt").read_text().strip())
print("Run directory:", RUN_DIR)
display(scores)

threshold_rows = []
for task, dimensions in report.get("pair", {}).items():
    for dimension, metrics in dimensions.items():
        threshold_rows.append({
            "task": task, "dimension": dimension,
            "test_tuned_threshold": metrics["best_threshold"],
            "test_accuracy": metrics["accuracy"],
            "test_macro_f1": metrics["f1"],
        })
if threshold_rows:
    print("\nThresholds tuned on the test set:")
    display(pd.DataFrame(threshold_rows))
            
print("\nSaved files:")
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(RUN_DIR)} ({path.stat().st_size / 1024:.1f} KiB)")

GSR / google-bert/bert-base-uncased
Commit: b349fe5710a6be49d38b26370f272d3ed9c1385d
Run directory: /content/drive/MyDrive/[Research Space]/[ICLR] Embedding MRL/gsr_bert_20260902_134835


,classification/banking77,classification/banking77:f1,classification/emotion,classification/emotion:f1,classification/tweet,classification/tweet:f1,pair/mrpc,pair/mrpc:f1,pair/scitail,pair/scitail:f1,...,sts/sts13,sts/sts14,sts/sts15,sts/sts16,sts/stsb,mean/classification,mean/classification_f1,mean/pair,mean/pair_f1,mean/sts
dim,,,,,,,,,,,,,,,,,,,,,
16,0.4746,0.4634,0.4990,0.3068,0.5629,0.5586,0.7026,0.6005,0.7408,0.7240,...,0.6239,0.5070,0.6055,0.6028,0.5526,0.5122,0.4430,0.6831,0.6434,0.5850
32,0.6616,0.6571,0.5332,0.3721,0.5912,0.5890,0.7159,0.6613,0.7531,0.7315,...,0.6566,0.5486,0.6602,0.6381,0.5944,0.5953,0.5394,0.6937,0.6682,0.6240
64,0.7760,0.7749,0.5785,0.4466,0.6579,0.6599,0.7304,0.6547,0.7596,0.7457,...,0.6782,0.5642,0.7006,0.6747,0.6347,0.6708,0.6272,0.7091,0.6783,0.6517
128,0.8469,0.8464,0.6088,0.4724,0.6774,0.6808,0.7316,0.6566,0.7733,0.7587,...,0.6924,0.5770,0.7220,0.6915,0.6546,0.7110,0.6665,0.7192,0.6887,0.6665
256,0.8761,0.8761,0.6400,0.5434,0.6920,0.6959,0.7339,0.6707,0.7888,0.7770,...,0.7031,0.5960,0.7415,0.6981,0.6736,0.7360,0.7051,0.7271,0.7014,0.6784
512,0.8800,0.8802,0.6490,0.5627,0.6973,0.7010,0.7333,0.6761,0.7902,0.7766,...,0.7135,0.6138,0.7548,0.7166,0.6859,0.7421,0.7146,0.7243,0.6984,0.6916
768,0.8891,0.8889,0.6536,0.5661,0.7010,0.7046,0.7380,0.6798,0.7926,0.7816,...,0.7173,0.6169,0.7512,0.7161,0.6847,0.7479,0.7199,0.7278,0.7030,0.6912



Thresholds tuned on the test set:


,task,dimension,test_tuned_threshold,test_accuracy,test_macro_f1
0,mrpc,dim_16,0.859296,0.702609,0.600529
1,mrpc,dim_32,0.884422,0.715942,0.661273
2,mrpc,dim_64,0.864322,0.730435,0.654667
3,mrpc,dim_128,0.869347,0.731594,0.656630
4,mrpc,dim_256,0.874372,0.733913,0.670655
5,mrpc,dim_512,0.914573,0.733333,0.676122
6,mrpc,dim_768,0.919598,0.737971,0.679847
7,scitail,dim_16,0.894472,0.740828,0.723997
8,scitail,dim_32,0.894472,0.753057,0.731454
9,scitail,dim_64,0.874372,0.759643,0.745748



Saved files:
  config.yaml (2.1 KiB)
  console.log (309.8 KiB)
  diagnostics/geometry_epoch2.json (7.5 KiB)
  diagnostics/geometry_epoch3.json (7.6 KiB)
  diagnostics/geometry_epoch4.json (7.6 KiB)
  diagnostics/geometry_epoch5.json (7.6 KiB)
  diagnostics/geometry_refresh_epoch2.json (7.5 KiB)
  diagnostics/geometry_refresh_epoch3.json (7.6 KiB)
  diagnostics/geometry_refresh_epoch4.json (7.6 KiB)
  diagnostics/geometry_refresh_epoch5.json (7.6 KiB)
  diagnostics/manifest.json (94.8 KiB)
  diagnostics/steps.jsonl (61.2 KiB)
  diagnostics/teacher_epoch2.json (23.7 KiB)
  diagnostics/teacher_epoch3.json (24.1 KiB)
  diagnostics/teacher_epoch4.json (24.1 KiB)
  diagnostics/teacher_epoch5.json (24.1 KiB)
  git_commit.txt (0.0 KiB)
  history.json (72.7 KiB)
  results.csv (1.6 KiB)
  results.json (21.0 KiB)
  results_epoch5.json (11.4 KiB)
  train.log (9.1 KiB)
